In [2]:
import tensorflow as tf
import gc
from tensorflow.keras import backend as K
import os

os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"

# Evitar que TensorFlow reserve toda la GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

print("GPU lista")
print(tf.config.list_physical_devices("GPU"))
def reset_tf():
    K.clear_session()
    gc.collect()

GPU lista
[]


In [3]:
from os import listdir
from numpy import asarray
from numpy import save
import tensorflow as tf
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
#Deshabilitar la GPU:
#tf.config.set_visible_devices([], 'GPU')
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array

folders = listdir('./yoga/')
#Clase Kirmizi será la clase 0.0
#Clase Siirt será la clase 1.0


photos =  []
labels = []

In [4]:
#2. IMPORTAMOS LOS DATOS:
for idx,folder in enumerate(folders):
    for file in listdir('./yoga/'+folder):
        #Cargamos la imagen.
        #load_img sirve para cargar las imágenes en memoria. Tiene distintos parámetros para modificar como se cargan las imágenes.
        photo = load_img('./yoga/'+folder+'/' + file, target_size=(16, 16)) 
        #Convertimos la imagen a un array.
        photo = img_to_array(photo)
        #Los guardamos en las listas.
        photos.append(photo)
        labels.append(float(idx))
        del photo
    print(idx)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37


In [5]:
photos = np.array(photos)
labels = np.array(labels)

In [6]:
# 3. Dividir entre train y test
X = photos
y = labels

# Normalizando Xz
X = X / 255.0

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
X_train.shape[1],

(16,)

In [8]:
# Haciendo la red neuronal a partir del tratamiento de PCA
from tensorflow import keras

model = keras.Sequential()
# Aplana las imágenes 16x16x3 a un vector 1D
model.add(keras.layers.Flatten(input_shape=(16, 16, 3)))
# Luego metemos capas ocultas
model.add(keras.layers.Dense(1024, activation="relu"))
model.add(keras.layers.Dense(768, activation="relu"))
# Luego metemos la capa de salida, que tiene 5 neuronas, una por cada clase, y función de activación softmax, que es la que se suele usar para clasificación multiclase.
model.add(keras.layers.Dense(38, activation="softmax"))

from tensorflow.keras import optimizers
sgd = optimizers.Adam(learning_rate=0.0001)

model.compile(loss="sparse_categorical_crossentropy", optimizer=sgd, metrics=["accuracy"])

/home/ciabd10/anaconda3/lib/python3.13/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [9]:
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=300, validation_split=0.1, callbacks=[early_stopping_cb])

Epoch 1/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.0366 - loss: 3.6738 - val_accuracy: 0.0652 - val_loss: 3.6071
Epoch 2/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1146 - loss: 3.5066 - val_accuracy: 0.0435 - val_loss: 3.5554
Epoch 3/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.1878 - loss: 3.4192 - val_accuracy: 0.1087 - val_loss: 3.4957
Epoch 4/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.2805 - loss: 3.3210 - val_accuracy: 0.2391 - val_loss: 3.3768
Epoch 5/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.3244 - loss: 3.2290 - val_accuracy: 0.1522 - val_loss: 3.3250
Epoch 6/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.3512 - loss: 3.1183 - val_accuracy: 0.3043 - val_loss: 3.2103
Epoch 7/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.3756 - loss: 3.0173 - val_accuracy: 0.1957 - val_loss: 3.1053
Epoch 8/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4683 - loss: 2.8952 - val_accuracy: 0.32

In [10]:
model.evaluate(X_test, y_test)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8860 - loss: 0.5030 


[0.5029996633529663, 0.8859649300575256]

In [11]:
del model
reset_tf()

## Red convolucional

In [12]:
print(X_train.shape) 

(456, 16, 16, 3)


In [13]:
from tensorflow import keras
from tensorflow.keras import layers, optimizers, initializers

# Definición de la red convolucional
model_cnn = keras.Sequential()

# Capa de convolución: el input_shape debe ser (128, 128, 1) (1 en blanco y negro, 3 en rgb)
model_cnn.add(keras.layers.Conv2D(16, (3, 3), activation="relu", input_shape=(16, 16, 3)))
model_cnn.add(keras.layers.MaxPooling2D((2, 2)))

model_cnn.add(keras.layers.Flatten())
model_cnn.add(keras.layers.Dense(768, activation="relu", kernel_initializer="he_normal"))
model_cnn.add(keras.layers.Dense(512, activation="relu", kernel_initializer="he_normal"))
model_cnn.add(keras.layers.Dense(38, activation="softmax", kernel_initializer="glorot_normal"))

# Optimizador y compilación
sgd_cnn = optimizers.Adam(learning_rate=0.0001)
model_cnn.compile(loss="sparse_categorical_crossentropy", optimizer=sgd_cnn, metrics=["accuracy"])

/home/ciabd10/anaconda3/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [14]:
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
history = model_cnn.fit(X_train, y_train, epochs=100, validation_split=0.1, callbacks=[early_stopping_cb])

Epoch 1/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.0317 - loss: 3.6723 - val_accuracy: 0.0435 - val_loss: 3.6279
Epoch 2/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.0756 - loss: 3.5813 - val_accuracy: 0.0435 - val_loss: 3.6114
Epoch 3/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1634 - loss: 3.5199 - val_accuracy: 0.1739 - val_loss: 3.5618
Epoch 4/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.2537 - loss: 3.4616 - val_accuracy: 0.1957 - val_loss: 3.5209
Epoch 5/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.3293 - loss: 3.3933 - val_accuracy: 0.2174 - val_loss: 3.4441
Epoch 6/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.3049 - loss: 3.3122 - val_accuracy: 0.3043 - val_loss: 3.3492
Epoch 7/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4098 - loss: 3.2151 - val_accuracy: 0.3043 - val_loss: 3.2674
Epoch 8/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4415 - loss: 3.1020 - val_accuracy: 0.3261 - 

In [15]:
model_cnn.evaluate(X_test, y_test)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9298 - loss: 0.4332 


[0.4332177937030792, 0.9298245906829834]

In [16]:
del model_cnn
reset_tf()

In [21]:
# Aplanando los datos de entrada de 28x28 a 784
X_train_reshape = X_train.reshape(X_train.shape[0], -1)
X_test_reshape = X_test.reshape(X_test.shape[0], -1)

# Asegúrate de que y_train y y_test sigan siendo vectores unidimensionales (en caso de que estén en formato 2D)
y_train_reshape = y_train.reshape(-1)
y_test_reshape = y_test.reshape(-1)

# Importando el clasificador Random Forest
from sklearn.ensemble import RandomForestClassifier

# Crear el modelo
clf = RandomForestClassifier(n_estimators=600, random_state=42, n_jobs=-1, bootstrap=False)

# Entrenar el modelo
clf.fit(X_train_reshape, y_train_reshape)

# Evaluando el modelo
y_pred = clf.predict(X_test_reshape)
accuracy = clf.score(X_test_reshape, y_test_reshape)
print(f'Accuracy: {accuracy}')

Accuracy: 0.8859649122807017
